# 24. 모델 학습 - 전체 데이터셋 (README §6)

**파이프라인 순서**

1. 데이터 로드 & 설정 확인  
2. 샘플링 전처리 (`AsymmetricSampler`)  
3. 기본 파라미터로 앙상블 학습 (빠른 검증)  
4. Optuna 하이퍼파라미터 튜닝 (선택)  
5. 최종 앙상블 학습 & 결과 요약  
6. 모델 저장  

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # notebooks/ 에서 실행 시

import numpy as np
import pandas as pd

from src.train_core import (
    SubsetTrainer, UnderbaggingEnsemble,
    run_training, print_ensemble_summary, plot_subset_prauc,
)
import config.train_config as cfg

print('환경 준비 완료')
print(f'TRAIN_PATH    : {cfg.TRAIN_PATH}')
print(f'VAL_TUNE_PATH : {cfg.VAL_TUNE_PATH}')

## 1. 데이터 로드 & 피처 확인


In [ ]:
df_train    = pd.read_parquet(cfg.TRAIN_PATH)
df_val_tune = pd.read_parquet(cfg.VAL_TUNE_PATH)

_meta = {'serial_number', 'date', 'failure', cfg.TARGET_COL}
FEATURE_COLS = cfg.FEATURE_COLS or [c for c in df_train.columns if c not in _meta]

pos_tr  = df_train[cfg.TARGET_COL].mean()
pos_val = df_val_tune[cfg.TARGET_COL].mean()
print(f'train    : {len(df_train):,} rows  pos_rate={pos_tr:.5f}')
print(f'val_tune : {len(df_val_tune):,} rows  pos_rate={pos_val:.5f}')
print(f'features : {len(FEATURE_COLS)} 개')

## 2. 빠른 검증 (기본 파라미터, Optuna 없이)


In [ ]:
result_quick = run_training(
    cfg=cfg,
    feature_cols=FEATURE_COLS,
    run_optuna=False,
    show_plots=True,
)

## 3. Optuna 하이퍼파라미터 튜닝


In [ ]:
# 시간이 오래 걸림 — 빠른 검증 후 결과가 만족스러우면 이 셀은 건너뛰어도 됨
result_tuned = run_training(
    cfg=cfg,
    feature_cols=FEATURE_COLS,
    run_optuna=True,
    optuna_rerank_delta=0.005,
    optuna_rerank_cap=5,
    optuna_n_trials=cfg.OPTUNA_N_TRIALS,
    optuna_timeout=cfg.OPTUNA_TIMEOUT,
    show_plots=True,
)

print('\nBest params:')
for k, v in result_tuned['best_params'].items():
    print(f'  {k}: {v}')

## 4. 모델 저장


In [ ]:
import joblib, json
from pathlib import Path

SAVE_DIR = Path(cfg.MODEL_SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 사용할 결과 선택 (Optuna 사용 시 result_tuned, 아니면 result_quick)
final_result = result_quick  # → result_tuned 로 교체 가능

# 서브셋 모델 저장
for i, model in enumerate(final_result['ensemble_result'].models):
    path = SAVE_DIR / f'subset_{i:02d}.pkl'
    joblib.dump(model, path)
    print(f'  저장: {path}')

# 피처 목록 저장
feat_path = SAVE_DIR / 'feature_cols.json'
with open(feat_path, 'w', encoding='utf-8') as f:
    json.dump(final_result['feature_cols'], f, ensure_ascii=False, indent=2)
print(f'  피처 목록: {feat_path}')

# best_params 저장
param_path = SAVE_DIR / 'best_params.json'
with open(param_path, 'w', encoding='utf-8') as f:
    json.dump(final_result['best_params'], f, ensure_ascii=False, indent=2)
print(f'  파라미터 : {param_path}')